# bebe-ia sur Colab — l'entrainement long

Le bebe apprend **en ligne, sans supervision de confiance**.
Objectif de ce notebook : des runs **1M, 10M interactions, 5 seeds**,
synchronisees avec le fragment GitHub via `scripts/synchroniser.py`.

Cellules : Drive -> clone -> install -> config -> donnees (Wikipedia + cache Drive)
-> boucle (GPU amp, tqdm) -> evaluation -> courbes -> checkpoints -> reprise.
**Reproductible : meme seed = memes resultats.**

In [ ]:
# 1. Drive (cache corpus + checkpoints survivent aux deconnexions)
from google.colab import drive
drive.mount('/content/drive')
import os
CACHE = '/content/drive/MyDrive/bebe_ia_cache'
os.makedirs(CACHE, exist_ok=True)
print('cache :', CACHE)

In [ ]:
# 2. Clone + install
%cd /content
!git clone https://github.com/ehwaza/bebe-ia.git
%cd /content/bebe-ia
!pip install -q -r requirements.txt tqdm
!ls

In [ ]:
# 3. Imports + config (SEEDS TRACEES — meme seed = memes resultats)
import sys, json, time
sys.path.insert(0, '/content/bebe-ia')
sys.path.insert(0, '/content/bebe-ia/scripts')
import torch
from entrainer_local import boucle_entrainement

CONFIG = {
    'n': 1_000_000,            # 1M par defaut ; 10M pour la grosse jambe
    'seeds': [1234, 1235, 1236, 1237, 1238],
    'source': 'wikipedia',
    'cache': CACHE,
    'eval_every': 5000,
    'ckpt_every': 100_000,
    'amp': torch.cuda.is_available(),
}
print(json.dumps({k: v for k, v in CONFIG.items() if k != 'cache'}, indent=2))
print('GPU :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# 4. Donnees — Wikipedia telechargee UNE fois, cachee sur Drive
from entrainer_local import charger_donnees, _charger_wikipedia
texte = _charger_wikipedia(CONFIG['cache'])
print('corpus :', len(texte), 'caracteres')
gen = charger_donnees('wikipedia', seed_data=1234 * 7919 + 13, cache=CONFIG['cache'])
for _ in range(3):
    src, cible = next(gen)
    print(repr(src), '->', repr(cible))

In [ ]:
# 5. Boucle d'entrainement (GPU amp + tqdm + checkpoints + reprise)
#    Une run = une seed. La reprise : relancer cette cellule, le dernier
#    checkpoint est repris via --reprendre (parent_dir).
from tqdm.notebook import tqdm

SEED = CONFIG['seeds'][0]        # <- change de seed pour chaque run
OUT = f'/content/drive/MyDrive/bebe_runs/seed_{SEED}'

resume = boucle_entrainement(
    n=CONFIG['n'], out_dir=OUT, seed=SEED,
    seed_data=SEED * 7919 + 13,          # unique par run, trace dans meta
    source=CONFIG['source'], cache=CONFIG['cache'],
    heldout_path='/content/bebe-ia/heldout.npz',
    eval_every=CONFIG['eval_every'], ckpt_every=CONFIG['ckpt_every'],
    amp=CONFIG['amp'],
    parent_dir=OUT + '/etat')             # reprise si le paquet existe
print(json.dumps(resume, indent=2))

In [ ]:
# 6. Evaluation + courbes (ECE, accuracy, risk-coverage)
import numpy as np, matplotlib.pyplot as plt
from bebe.metriques import resume_metriques

z = np.load(OUT + '/hist.npz')
conf, ok, ece = z['conf'], z['ok'], z['ece']
print(json.dumps(resume_metriques(conf, ok), indent=2))

fig, axs = plt.subplots(1, 2, figsize=(14, 4))
axs[0].plot(ece); axs[0].set_xlabel('evaluation x CONFIG["eval_every"]')
axs[0].set_ylabel('ECE held-out'); axs[0].set_title('Calibration')
axs[1].plot(np.cumsum(ok) / np.arange(1, len(ok) + 1))
axs[1].set_xlabel('interactions'); axs[1].set_ylabel('accuracy cumulee')
axs[1].set_title('Apprentissage')
for ax in axs: ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(OUT + '/courbes.png', dpi=120); plt.show()

In [ ]:
# 7. Synchronisation avec le fragment GitHub (fusion par confiance)
#    A lancer depuis le clone bebe-ia, avec les etats locaux copies dans etats/.
!cp -r {OUT}/etat /content/bebe-ia/etats/local
!cd /content/bebe-ia && git pull --rebase --autostash || true
!cd /content/bebe-ia && python scripts/synchroniser.py
# Pour uploader (necessite un token avec droits d'ecriture) :
# !cd /content/bebe-ia && python scripts/synchroniser.py --push